# 2. Find the interpreter

For every broadcast of `work/broadcasts.json`: download it, find the frame ranges in which the
interpreter is on screen, and keep what the later steps need: the audio of each range (for
4_labels) and its first and last frames (for 3_verify). Then the download is deleted.

- **PrimaTV and ProTV** interpret one block of 25 to 30 minutes per broadcast. It is found by
  face recognition of the channel's interpreters (`known_people/`): a frame every 15 minutes
  until one shows a known interpreter, then a binary search for the block's first and last frame.
  A second search looks for interpretation that the first one did not see.
- **Digi24** scatters interpretation over the journal. A small classifier marks every frame
  with or without the interpreter, and a state machine turns that into ranges.

Output, `work/segments.json`: for every broadcast its interpreter, its height and the ranges
[start, end) as frame indices of the broadcast (25 fps). Also `work/audio/` and `work/ends/`.
The original run's ranges are in `frozen/segments.json`.

In [ ]:
import math
import sys

import numpy as np

from common import CROP, DECISIONS, FPS, FROZEN, REPO, WORK, Classifier, Faces, Video, digi24_era, first_true, read, \
    save_audio, write

sys.path.insert(0, str(REPO / 'scripts'))   # the downloaders' stream lookups

COOKIES = 'cookies.txt'            # a logged-in YouTube session's cookies (PrimaTV; also needs the PO-token server, see README)
DOWNLOADS = WORK / 'downloads'     # one broadcast at a time
OUT = WORK / 'segments.json'
broadcasts = read(WORK / 'broadcasts.json')
segments = read(OUT) if OUT.exists() else {'PrimaTV': {}, 'ProTV': {}, 'Digi24': {}}


def keep(channel, key, video_path, interpreter, ranges, **notes):
    """Record a broadcast's ranges (None: no interpreter found) and save, for each range, its audio
    and its first and last 3 frames, cropped as released."""
    if ranges is None:
        segments[channel][key] = None
        write(OUT, segments)
        return
    video = Video(video_path)
    crop = CROP[channel][digi24_era(key) if channel == 'Digi24' else video.height]
    ends = {}
    for i, (start, end) in enumerate(ranges):
        save_audio(video_path, start, end, WORK / 'audio' / channel / key / f'{i}.flac')
        for j in [*range(start, start + 3), *range(end - 3, end)]:
            ends[f'{i}/{j}'] = video[j][crop['y']:crop['y'] + crop['h'], crop['x']:crop['x'] + crop['w']]
    np.savez_compressed(WORK / 'ends' / channel / f"{key.replace('/', '_')}.npz", **ends)
    segments[channel][key] = {'interpreter': interpreter, 'source_height': video.height, 'segments': ranges, **notes}
    write(OUT, segments)


for channel in segments:
    (WORK / 'ends' / channel).mkdir(parents=True, exist_ok=True)

## Face search (PrimaTV, ProTV)

`FACE_BOX` is where the face is looked for, by frame height: the notebooks' interpreter box
(numpy slicing clips it at the frame's edge). The released crops are `common.CROP`.

In [ ]:
FACE_BOX = {'PrimaTV': {1080: (1522, 720, 383, 411), 720: (1015, 480, 255, 274)},
            'ProTV': {720: (952, 347, 300, 322), 432: (571, 208, 180, 193)}}
INTERPRETERS = {'PrimaTV': ['interpreter_03', 'interpreter_04', 'interpreter_05'],
                'ProTV': ['interpreter_01', 'interpreter_02']}


def finder(channel, video):
    """who(i, shrink=False): the interpreter recognized in frame i's box, or None."""
    import cv2
    faces = Faces(INTERPRETERS[channel])
    x, y, w, h = FACE_BOX[channel][video.height]

    def who(i, shrink=False):
        box = video[i][y:y + h, x:x + w]
        if shrink:   # ProTV's quicker check
            box = cv2.resize(box, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_AREA)
        return faces.who(box)
    return who


def find_block(who, frames, start=0, stop=None, stride=15 * 60 * FPS):
    """The block of frames showing a known interpreter: probe every `stride` frames of
    [start, stop) until one does, then binary-search back for the first such frame and forward
    for the last. -> (first, last, interpreter), last inclusive, or None."""
    for i in range(start, frames if stop is None else stop, int(stride)):
        interpreter = who(i)
        if interpreter:
            break
    else:
        return None
    first = first_true(0, i, lambda x: who(x) is not None)
    gone = first_true(i, frames - 1, lambda x: who(x) is None)
    last = gone - 1 if gone < frames and who(gone) is None else frames - 1
    return first, last, interpreter

### PrimaTV

The PrimaTV search ignores the video's last frame and probes at least 4 frames. Then it
looks again, every 5 minutes, before and after the block. A video where it finds the
interpreter again is marked `look_again` for a human to decide; the original run's two
(fjYZNr8KjGE, ta_GxiXxbqw) were dropped, see `decisions/primatv.json`.

In [ ]:
from yt_dlp import YoutubeDL

for vid in broadcasts['PrimaTV']:
    if vid in segments['PrimaTV']:
        continue
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    with YoutubeDL({'format': 'bv*+ba[language=ro]/bv*+ba/b', 'merge_output_format': 'mkv', 'quiet': True,
                    'outtmpl': str(DOWNLOADS / '%(id)s.%(ext)s'), 'retries': 10, 'cookiefile': COOKIES,
                    'js_runtimes': {'node': {}}, 'remote_components': ['ejs:github']}) as ydl:
        ydl.download([f'https://www.youtube.com/watch?v={vid}'])
    video_path = next(DOWNLOADS.glob(f'{vid}.*'))
    video = Video(video_path)
    who = finder('PrimaTV', video)
    frames = video.frames - 1
    block = find_block(who, frames, stride=min(15 * 60 * FPS, frames // 4))
    if block:
        first, last, interpreter = block
        stride = min(5 * 60 * FPS, frames // 12)
        again = [side for side, lo, hi in (('before', 0, first - 1), ('after', last + 1, frames))
                 if find_block(who, frames, lo, hi, stride)]
        keep('PrimaTV', vid, video_path, interpreter, [[first, last + 1]],
             **({'look_again': 'interpreter seen again ' + ' and '.join(again)} if again else {}))
    else:
        keep('PrimaTV', vid, video_path, None, None)
    video_path.unlink()

### ProTV

Up to 2024-02-29 the broadcasts come from protv.ro, at the highest height up to 720p; from
2024-03-01 from vid2, at 432p, as in the original run. While vid2 serves no files they come
from protv.ro's 432p version, which has the same frames (3_verify found 249 of the original vid2
broadcasts there unchanged). vid2 is downloaded at about 1 MB/s, as the original run did.

After the block is found, the 10 minutes after its end are probed every 5 s: if the interpreter
is back, the block goes on there, up to its next disappearance. Otherwise the block's last 10
minutes are probed every 5 s backwards: if the interpreter is gone for most of the 3 seconds
there, the block is split around that gap.

In [ ]:
import time

import requests
from download_protv import resolve_manifest_url

VID2_FROM = '2024-03-01'


def protv_ranges(who, frames, first, last):
    """[[start, end), ...] of a ProTV broadcast whose face search found [first, last]."""
    for i in range(last + 1, min(last + 1 + 10 * 60 * FPS, frames), 5 * FPS):
        if who(i, shrink=True):          # the interpreter is back after the block
            gone = first_true(i, frames - 1, lambda x: who(x, shrink=True) is None)
            if gone < frames and who(gone) is None:
                return [[first, last + 1], [i, gone]]
            return [[first, last + 1]]
    for i in range(last - 1, max(first, last - 1 - 10 * 60 * FPS), -5 * FPS):
        if not who(i, shrink=True):      # a gap in the block's last 10 minutes
            probes = range(max(first, i - 1), max(first, int(i - 1 - 3 * FPS)) - 1, -5)
            if sum(who(j) is None for j in probes) * 2 < len(probes):
                break                    # mostly there: not a gap
            gone = first_true(first, i, lambda x: who(x) is None)
            back = first_true(i, last, lambda x: who(x) is not None)
            if gone < frames and who(gone) is None:
                return [[first, gone], [back, last + 1]]
            break
    return [[first, last + 1]]


def download_protv(route, url, height):
    """A ProTV broadcast -> DOWNLOADS/protv.(mp4|mkv)"""
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    if route == 'vid2':
        path = DOWNLOADS / 'protv.mp4'
        with requests.get(url, stream=True, timeout=60) as r, open(path, 'wb') as f:
            r.raise_for_status()
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
                time.sleep(1)
        return path
    form = f'bv[height<={height}]+ba/b[height<={height}]/b' if height == 720 else f'bv[height={height}]+ba'
    with YoutubeDL({'format': form, 'merge_output_format': 'mkv', 'quiet': True,
                    'outtmpl': str(DOWNLOADS / 'protv.%(ext)s'), 'retries': 10}) as ydl:
        ydl.download([resolve_manifest_url(requests.Session(), url)])
    return DOWNLOADS / 'protv.mkv'


episode = lambda url: url.rstrip('/').rsplit('/', 1)[1].split('-')[0]
pages, files = broadcasts['ProTV']['protv.ro'], broadcasts['ProTV']['vid2']
jobs = [(f'{day}/{episode(u)}', 'protv.ro', u, 720) for day, us in pages.items() if day < VID2_FROM for u in us]
if files:
    jobs += [(f"{day}/{u.rsplit('/', 1)[1][:-4]}", 'vid2', u, 432) for day, us in files.items() for u in us]
else:
    jobs += [(f'{day}/{episode(u)}', 'protv.ro', u, 432) for day, us in pages.items() if day >= VID2_FROM for u in us]

for key, route, url, height in jobs:
    if key in segments['ProTV']:
        continue
    try:
        video_path = download_protv(route, url, height)
    except Exception as error:
        print(key, 'not downloaded:', error)
        continue
    video = Video(video_path)
    who = finder('ProTV', video)
    block = find_block(who, video.frames)
    if block:
        first, last, interpreter = block
        keep('ProTV', key, video_path, interpreter, protv_ranges(who, video.frames, first, last))
    else:
        keep('ProTV', key, video_path, None, None)
    video_path.unlink()

## Digi24

### Which days have an interpreter

A pose check on the interpreter box: exactly one person, facing the camera (nose, eyes and
shoulders seen, shoulders level within 20%), both elbows and wrists and both hips seen
(RTMW whole-body pose, confidence 0.5). A day has an interpreter if more than half of the
frames every 4 s from minute 5 to minute 15 pass it.

In [ ]:
import cv2
import requests
import torch
from rtmlib import Wholebody

from download_digi24 import USER_AGENT

pose = Wholebody(mode='lightweight', backend='onnxruntime', device='cuda' if torch.cuda.is_available() else 'cpu')
COCO = {'nose': 0, 'left_eye': 1, 'right_eye': 2, 'left_shoulder': 5, 'right_shoulder': 6, 'left_elbow': 7,
        'right_elbow': 8, 'left_wrist': 9, 'right_wrist': 10, 'left_hip': 11, 'right_hip': 12}


def looks_like_interpreter(image):
    keypoints, scores = pose(image)
    if keypoints.shape[0] != 1:
        return False
    k, s = keypoints[0], scores[0]
    seen = lambda *parts: all(s[COCO[p]] >= 0.5 for p in parts)
    y_left, y_right = k[COCO['left_shoulder'], 1], k[COCO['right_shoulder'], 1]
    return (seen('nose', 'left_eye', 'right_eye', 'left_shoulder', 'right_shoulder')
            and abs(y_left - y_right) <= 0.2 * (y_left + y_right)
            and seen('left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip'))


def pose_box(frame, day):
    """The interpreter box of the pose check (fractions of the 1280x720 frame)."""
    if digi24_era(day) == 'from 2023-12':
        return frame[int(0.400 * 720):int(0.705 * 720), int(0.755 * 1280):int(0.950 * 1280)]
    return frame[int(0.660 * 720):int(0.945 * 720), int(0.800 * 1280):int(0.945 * 1280)]


def has_interpreter(video_path, day):
    cap = cv2.VideoCapture(str(video_path))
    total, checks = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), []
    for i in range(5 * 60 * FPS, min(15 * 60 * FPS, total - 1) + 1, 4 * FPS):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ok, frame = cap.read()
        if not ok:
            break
        checks.append(looks_like_interpreter(pose_box(frame, day)))
    return bool(checks) and sum(checks) / len(checks) > 0.5


def download_digi24(day):
    """The day's journal (720p) -> DOWNLOADS/digi24.mp4."""
    pages = broadcasts['Digi24']['listing'][day]
    url = read(DECISIONS / 'digi24.json')['pages'].get(day, pages[0])
    if len(pages) > 1 and url == pages[0]:
        print(day, 'has', len(pages), 'pages; taking the first: add the right one to decisions/digi24.json')
    src = broadcasts['Digi24']['video_src'][url]
    DOWNLOADS.mkdir(parents=True, exist_ok=True)
    path = DOWNLOADS / 'digi24.mp4'
    headers = {'User-Agent': USER_AGENT, 'Referer': src, 'Accept-Encoding': 'identity',
               'Accept': 'video/webm,video/ogg,video/*;q=0.9,application/ogg;q=0.7,audio/*;q=0.6,*/*;q=0.5'}
    with requests.get(src, stream=True, timeout=60, headers=headers) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(8 << 20):
                f.write(chunk)
    return path

### The classifier

Trained on the days with an interpreter: 40 frames of each, spread evenly, labelled by the pose
check above and padded to 250x219. The head of an ImageNet MobileNetV3-Small is trained
(backbone frozen), the minority class augmented and the classes weighted, until 99.5% of the
training frames are right. The original run's weights are `frozen/models/digi24_classifier.pt`;
set `TRAIN = True` to train new ones (this needs every day's download once more).

In [ ]:
TRAIN = False
WEIGHTS = FROZEN / 'models' / 'digi24_classifier.pt'
days = [d for d in broadcasts['Digi24']['listing'] if d not in segments['Digi24']]

if TRAIN:
    import itertools
    import torch
    from PIL import Image
    from torch import nn
    from torch.utils.data import ConcatDataset, DataLoader, Dataset
    from torchvision import models, transforms as T

    def pad(frame, w=250, h=219):
        top, left = (h - frame.shape[0]) // 2, (w - frame.shape[1]) // 2
        return np.pad(frame, ((top, h - frame.shape[0] - top), (left, w - frame.shape[1] - left), (0, 0)))

    with_interpreter, without = [], []
    for day in days:
        path = download_digi24(day)
        if has_interpreter(path, day):
            cap = cv2.VideoCapture(str(path))
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            for i in range(0, total, math.ceil(total / 40)):
                cap.set(cv2.CAP_PROP_POS_FRAMES, i)
                ok, frame = cap.read()
                if not ok:
                    break
                box = pose_box(frame, day)
                (with_interpreter if looks_like_interpreter(box) else without).append(pad(box))
        path.unlink()

    weights = models.MobileNet_V3_Small_Weights.DEFAULT
    mean, std = weights.meta.get('mean', (0.485, 0.456, 0.406)), weights.meta.get('std', (0.229, 0.224, 0.225))

    class Frames(Dataset):
        def __init__(self, frames, label, augment):
            h, w = frames[0].shape[:2]
            extra = [T.ColorJitter(0.1, 0.1, 0.1, 0.05), T.RandomRotation(5),
                     T.RandomResizedCrop((h, w), scale=(0.9, 1.0))] if augment else []
            self.frames, self.label, self.tfm = frames, label, T.Compose(extra + [T.ToTensor(), T.Normalize(mean, std)])

        def __len__(self):
            return len(self.frames)

        def __getitem__(self, i):
            return self.tfm(Image.fromarray(self.frames[i][..., ::-1])), self.label

    minority = len(with_interpreter) < len(without)
    data = DataLoader(ConcatDataset([Frames(with_interpreter, 1, minority), Frames(without, 0, not minority)]),
                      batch_size=512, shuffle=True, num_workers=4, pin_memory=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
    for p in itertools.chain(model.features.parameters(), model.avgpool.parameters()):
        p.requires_grad = False
    model.to(device)
    counts = torch.tensor([len(without), len(with_interpreter)], dtype=torch.float, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=counts.sum() / (2 * counts))
    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3)
    best, patience = 0.0, 3
    for epoch in range(6):
        model.train()
        for x, y in data:
            loss = loss_fn(model(x.to(device)), y.to(device))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            right = sum((model(x.to(device)).argmax(1).cpu() == y).sum().item() for x, y in data)
        accuracy = right / len(data.dataset)
        print(f'epoch {epoch + 1}: {accuracy:.4f}')
        best, patience = (accuracy, 3) if accuracy > best else (best, patience - 1)
        if patience == 0 or best >= 0.995:
            break
    WEIGHTS = WORK / 'digi24_classifier.pt'
    torch.save(model.state_dict(), WEIGHTS)

### The ranges

The classifier sees every frame of the box. A range opens once the interpreter has been seen
for 2 s and closes once they have been gone for 2 s; ranges under 10 s are dropped. The end
kept is the last frame seen, as an exclusive end: the original run cut every range up to, not
including, that frame, and the released bounds keep it so.

In [ ]:
CLASSIFIER_BOX = {'until 2023-11': (1018, 1210, 474, 680), 'from 2023-12': (966, 1216, 287, 510)}  # x0, x1, y0, y1
classify = Classifier(WEIGHTS, pad_to=(250, 223))


def digi24_ranges(video_path, day):
    x0, x1, y0, y1 = CLASSIFIER_BOX[digi24_era(day)]
    ranges, start, gone, maybe = [], -1, -1, -1
    index = -1
    for batch in Video(video_path).read():
        for seen in classify([f[y0:y1, x0:x1] for f in batch]):
            index += 1
            if seen:
                if start >= 0:
                    gone = -1
                else:
                    maybe = index if maybe < 0 else maybe
                    if index - maybe >= 2 * FPS:
                        start, gone = maybe, -1
                continue
            maybe = -1
            if start >= 0 and gone < 0:
                gone = index
            if gone >= 0 and index - gone >= 2 * FPS:
                if gone - 1 - start >= 10 * FPS:
                    ranges.append([start, gone - 1])
                start, gone = -1, -1
    if start >= 0 and index - start >= 10 * FPS:
        ranges.append([start, index])
    return ranges


for day in days:
    video_path = download_digi24(day)
    keep('Digi24', day, video_path, None, digi24_ranges(video_path, day) if has_interpreter(video_path, day) else None)
    video_path.unlink()